<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Solution: Train a tiny language model

**[Train a tiny language model](https://learning.nextia-ai.com/courses/deep-learning/m09/train-a-tiny-language-model/)** · Deep Learning and Transformers Explained · Module 9, case study 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** train a small character-level transformer from scratch on six Shakespeare tragedies, watch its samples improve during training, compare it with two counting baselines on held-out text, try the temperature, and reload the saved model in a new process to write from your own prompt.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 3 minutes on the course's Mac mini (2 CPU threads); a slower computer, such as free Colab, can take two or three times as long |
| **Needs** | A browser and a free Colab or Kaggle account. The setup cell downloads the text (about 0.9 MB). In Kaggle, turn on Internet in the notebook settings. |
| **You should know** | The causal mask, from [Position and masking](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/); next-token prediction, from [Training objectives](https://learning.nextia-ai.com/courses/deep-learning/m06/training-objectives/); the generation loop and temperature, from [Training versus inference](https://learning.nextia-ai.com/courses/deep-learning/m07/training-versus-inference/). |
| **Tested** | Python 3.12 with PyTorch 2.14.1 (CPU, 2 threads), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m09/train-a-tiny-language-model/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M09-L04-train-a-tiny-language-model/train-a-tiny-language-model-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

## Setup: get the text

Run the next cell. It makes a folder `tiny-lm`, downloads the text of six Shakespeare tragedies (about 0.9 MB) from the course's labs repository, and checks its checksum. It needs an internet connection and no account.

You should see `Ready: 850,875 characters.` If the cell shows `wrong checksum`, delete the folder `tiny-lm` and run the cell again.

The text is public domain. It comes from Project Gutenberg eBook #100, *The Complete Works of William Shakespeare*; the dataset card `dataset.md` next to this notebook gives the details.

In [ ]:
# Setup: download the text into tiny-lm/ and check it.
import hashlib, os, subprocess, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
URL = LABS + "M09-L04-train-a-tiny-language-model/shakespeare_tragedies.txt"
SHA256 = "59bd62e67e9720ca07b12d371b0311c858a0198c4a27466b382d3697daac94b4"
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
Path("tiny-lm").mkdir(exist_ok=True)
os.chdir("tiny-lm")
Path(".nextia-notebook").touch()
path = Path("shakespeare_tragedies.txt")
if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != SHA256:
    try:
        urllib.request.urlretrieve(URL, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), URL]).returncode:
            raise SystemExit(f"Could not download {URL}. Check your internet connection.")
    if hashlib.sha256(path.read_bytes()).hexdigest() != SHA256:
        raise SystemExit("shakespeare_tragedies.txt has the wrong checksum. Delete the folder tiny-lm and run again.")
print(f"Ready: {len(path.read_text(encoding='utf-8')):,} characters.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The data

Load the text and print the first lines of *Hamlet*. Run the cell.

In [ ]:
text = Path("shakespeare_tragedies.txt").read_text(encoding="utf-8")
print(f"{len(text):,} characters, {len(text.splitlines()):,} lines")
start = text.index("BARNARDO.\nWho’s there?")
print(text[start:start + 196])

> **Check.** You should see `850,875 characters, 33,143 lines`, then Barnardo's "Who’s there?" and Francisco's answer.

Next, the characters and the shape of the text: how many different characters, which are common, and how many lines are a speaker's name. Run the cell.

In [ ]:
import re
from collections import Counter

counts = Counter(text)
print(len(counts), "different characters")
print("most common:", [(c, n) for c, n in counts.most_common(8)])
print("rarest:     ", [(c, n) for c, n in counts.most_common()[-5:]])
SPEAKER = re.compile(r"^([A-Z][A-Z’ ]+)\.$", flags=re.M)      # a line like "HAMLET."
print("speaker lines:", len(SPEAKER.findall(text)), "| different speakers:", len(set(SPEAKER.findall(text))))

> **Check.** You should see `75 different characters`; the space and the letter `e` are the most common. There are 5,647 speaker lines and 184 different speakers.

## 2. Split the text: one cut, no shuffling

The first 90% of the text is for training, the last 10% for validation. Run the cell.

In [ ]:
cut = int(len(text) * 0.9)
train_text, valid_text = text[:cut], text[cut:]
print(f"train {len(train_text):,} characters | valid {len(valid_text):,} characters")
print("valid starts:", repr(valid_text[:60]))

> **Check.** You should see `train 765,787 characters | valid 85,088 characters`. Validation starts in the middle of a line of *Romeo and Juliet*.

## 3. Characters as tokens

The vocabulary is every character of the training text. Each character gets an ID, its position in the sorted list. Run the cell.

In [ ]:
chars = sorted(set(train_text))                 # the vocabulary: every character of the training text
stoi = {c: i for i, c in enumerate(chars)}      # character -> ID


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(chars[i] for i in ids)


print(len(chars), "characters in the vocabulary")
print("in valid but not in train:", sorted(set(valid_text) - set(chars)))
print(repr("".join(chars)))
print(encode("ROMEO."), "->", repr(decode(encode("ROMEO."))))
train_ids = torch.tensor(encode(train_text))
valid_ids = torch.tensor(encode(valid_text))
show_shape("train_ids", train_ids)
show_shape("valid_ids", valid_ids)

> **Check.** You should see 75 characters, an empty list for "in valid but not in train", and the shapes `(765787,)` and `(85088,)`.

> **Your turn.** Put the IDs of the text `"ROMEO."` in `romeo_ids`, as a list of numbers. Use `encode()`. Then run the check cell.

In [ ]:
romeo_ids = encode("ROMEO.")
print(romeo_ids)

In [ ]:
expect_hash('romeo_ids', romeo_ids, 'ef9ec042f5fd5678')

## 4. Baselines: what does counting achieve?

Three baselines, each measured as the average loss per character on the validation text (lower is better). Run the cell.

In [ ]:
import math

V = len(chars)
print("1. every character equally likely:", round(math.log(V), 3))

counts = torch.bincount(train_ids, minlength=V).float() + 1          # +1: no character has chance 0
p_char = counts / counts.sum()
print("2. character frequencies:         ", round(float(-p_char[valid_ids].log().mean()), 3))

pairs = torch.ones(V, V)                                              # +1 for every pair, as above
pairs.index_put_((train_ids[:-1], train_ids[1:]), torch.ones(len(train_ids) - 1), accumulate=True)
p_next = pairs / pairs.sum(1, keepdim=True)                           # row = this character, column = next
bigram_loss = float(-p_next[valid_ids[:-1], valid_ids[1:]].log().mean())
print("3. pair counts (bigram):          ", round(bigram_loss, 3))

> **Check.** You should see about 4.317, 3.313 and 2.499.

What does the pair-count model write? Run the cell.

In [ ]:
def bigram_generate(prompt, length=200, seed=0):
    g = torch.Generator().manual_seed(seed)
    ids = encode(prompt)
    for _ in range(length):
        ids.append(int(torch.multinomial(p_next[ids[-1]], 1, generator=g)))
    return decode(ids)


print(bigram_generate("ROMEO.\n"))

## 5. The model

A decoder built from the course's parts: a character table, a position table, 3 transformer blocks with a causal mask, and an output layer. Run the cell. It prints the number of parameters of each part.

In [ ]:
from torch import nn

class TinyLM(nn.Module):
    """Character + position vectors -> blocks with a causal mask -> a score for every character."""

    def __init__(self, vocab_size, dim=128, heads=4, blocks=3, context=64, dropout=0.1):
        super().__init__()
        self.context = context
        self.embed = nn.Embedding(vocab_size, dim)            # one row of numbers per character
        self.position = nn.Embedding(context, dim)            # one row of numbers per position
        block = nn.TransformerEncoderLayer(dim, heads, 4 * dim, dropout=dropout,
                                           batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(block, blocks, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(dim)
        self.out = nn.Linear(dim, vocab_size)                 # a score for every character

    def forward(self, ids):                                   # ids: (batch, n)
        n = ids.shape[1]
        h = self.embed(ids) + self.position(torch.arange(n))  # (batch, n, dim)
        mask = nn.Transformer.generate_square_subsequent_mask(n)
        h = self.blocks(h, mask=mask, is_causal=True)         # each position reads only the past
        return self.out(self.norm(h))                         # (batch, n, vocab_size)


torch.manual_seed(0)
model = TinyLM(V)
for name, part in [("character table", model.embed), ("position table", model.position),
                   ("3 blocks", model.blocks), ("final norm", model.norm), ("output layer", model.out)]:
    print(f"{name:16s} {sum(p.numel() for p in part.parameters()):>8,}")
print(f"{'total':16s} {sum(p.numel() for p in model.parameters()):>8,}")

> **Check.** You should see `total 622,539`.

> **Your turn.** How many parameters do the character table and the position table have together? Each has one row of 128 numbers per character (75) or per position (64). Compute it by hand, as a number, in `table_params`. Then run the check cell.

In [ ]:
table_params = 75 * 128 + 64 * 128
print(table_params)

In [ ]:
expect('table_params', table_params, 17792)

A training batch is 32 random windows of 64 characters; the answers are the same windows moved one character to the right. Run the cell.

In [ ]:
def get_batch(ids, generator, batch_size=32, context=64):
    """batch_size random windows of the text, and the same windows moved one character to the right."""
    starts = torch.randint(len(ids) - context - 1, (batch_size,), generator=generator)
    x = torch.stack([ids[s:s + context] for s in starts])
    y = torch.stack([ids[s + 1:s + context + 1] for s in starts])
    return x, y


x, y = get_batch(train_ids, torch.Generator().manual_seed(0))
show_shape("x", x)
show_shape("y", y)
print("x[0]:", repr(decode(x[0].tolist())[:40]))
print("y[0]:", repr(decode(y[0].tolist())[:40]))
loss_fn = nn.CrossEntropyLoss()
with torch.no_grad():
    scores = model(x)
show_shape("scores", scores)
print("untrained loss:", round(loss_fn(scores.reshape(-1, V), y.reshape(-1)).item(), 3), "| ln 75 =", round(math.log(V), 3))

> **Check.** You should see `x shape (32, 64)`, `scores shape (32, 64, 75)` and an untrained loss of about 4.4, close to ln 75 = 4.317.

The generation loop: score the next character, choose one, add it, repeat. Run the cell. The model is untrained, so expect noise.

In [ ]:
@torch.no_grad()
def generate(model, prompt, length=200, temperature=0.8, seed=0):
    """The generation loop: score the next character, choose one, add it, repeat."""
    model.eval()
    g = torch.Generator().manual_seed(seed)
    ids = encode(prompt)
    for _ in range(length):
        scores = model(torch.tensor([ids[-model.context:]]))[0, -1]    # the last position's scores
        if temperature == 0:
            next_id = int(scores.argmax())                              # greedy
        else:
            probs = torch.softmax(scores / temperature, dim=-1)
            next_id = int(torch.multinomial(probs, 1, generator=g))
        ids.append(next_id)
    return decode(ids)


print(generate(model, "ROMEO.\n", length=80))

## 6. Train, and watch the samples

> **Predict.** At which step will the model first write lines that look like a play: a name in capitals on its own line, then a speech? Write your guess, then run the cell. It trains 3,000 steps: about 3 minutes on the course's Mac mini, and up to two or three times as long on a slower computer such as free Colab.

In [ ]:
import time

STEPS, EVAL_EVERY, SAMPLE_AT = 3000, 100, (0, 300, 1000, 3000)


@torch.no_grad()
def estimate(model, ids, batches=20):
    """Mean loss on 20 fixed random batches (the same batches every time)."""
    model.eval()
    g = torch.Generator().manual_seed(1)
    losses = [loss_fn(model(x).reshape(-1, V), y.reshape(-1)).item()
              for x, y in (get_batch(ids, g) for _ in range(batches))]
    return sum(losses) / len(losses)


torch.manual_seed(0)
model = TinyLM(V)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
batches = torch.Generator().manual_seed(0)
history, samples = [], {}
started = time.perf_counter()
for step in range(STEPS + 1):
    if step % EVAL_EVERY == 0:
        history.append({"step": step, "train": round(estimate(model, train_ids), 4),
                        "valid": round(estimate(model, valid_ids), 4)})
        if step % 500 == 0:
            print(history[-1], f"{time.perf_counter() - started:.0f} s")
    if step in SAMPLE_AT:
        samples[step] = generate(model, "ROMEO.\n", length=1000, temperature=0.8, seed=0)
    if step == STEPS:
        break
    model.train()                                  # dropout on for training
    x, y = get_batch(train_ids, batches)
    optimizer.zero_grad()
    loss = loss_fn(model(x).reshape(-1, V), y.reshape(-1))
    loss.backward()
    optimizer.step()
print(f"trained {STEPS} steps in {time.perf_counter() - started:.0f} s")

> **Check.** The last row should have a validation loss of about 1.6, and the training loss a little lower. On another computer the numbers can differ a little in the last digits.

Now read the samples, all from the same prompt and the same seed. Run the cell.

In [ ]:
for step, sample in samples.items():
    print(f"--- step {step} ---")
    print(sample[:230])

## 7. The fair comparison

All four on the whole validation text, under one metric. Run the cell.

In [ ]:
@torch.no_grad()
def full_loss(model, ids):
    """Loss on the whole text, in windows of 64 characters, with dropout off."""
    model.eval()
    n = (len(ids) - 1) // model.context * model.context
    x, y = ids[:n].view(-1, model.context), ids[1:n + 1].view(-1, model.context)
    total = sum(loss_fn(model(xb).reshape(-1, V), yb.reshape(-1)).item() * yb.numel()
                for xb, yb in zip(x.split(256), y.split(256)))
    return total / n


results = {"every character equally likely": math.log(V),
           "character frequencies": float(-p_char[valid_ids].log().mean()),
           "pair counts (bigram)": bigram_loss,
           "tiny transformer": full_loss(model, valid_ids)}
for name, loss in results.items():
    print(f"{name:32s} valid loss {loss:.3f}  ~ a choice among {math.exp(loss):4.1f} characters")
print(f"{'tiny transformer, training text':32s} loss {full_loss(model, train_ids):.3f}")

> **Check.** The tiny transformer should have the lowest validation loss, about 1.585, against 2.499 for the pair counts.

## 8. Temperature

The same prompt and seed at four temperatures. Run the cell.

In [ ]:
for t in (0, 0.5, 1.0, 1.5):
    print(f"--- temperature {t} ---")
    print(generate(model, "LEAR.\n", length=160, temperature=t, seed=0))

The model's own probabilities for the character after `Give me the `. Run the cell.

In [ ]:
prompt = "LEAR.\nGive me the "
with torch.no_grad():
    model.eval()
    scores = model(torch.tensor([encode(prompt)]))[0, -1]
probs = torch.softmax(scores, dim=-1)
for p, i in zip(*probs.topk(8)):
    print(f"{chars[i]!r:6s} {p.item():.3f}")

> **Your turn.** Write 200 greedy characters (temperature 0) from the prompt `"LEAR.\n"` into `greedy`. Then run the check cell. It checks the length, not the text.

In [ ]:
greedy = generate(model, "LEAR.\n", length=200, temperature=0)
print(greedy)

In [ ]:
expect("the length of greedy", len(greedy) if isinstance(greedy, str) else greedy, 206)

## 9. What it gets right, and what it gets wrong

A few checks that a reader makes first: are the words real words, how many different words are there, are the speaker names real names? Run the cell.

In [ ]:
train_words = set(re.findall(r"[a-z’]+", train_text.lower()))
speakers = set(SPEAKER.findall(train_text))


def judge(sample):
    """What a reader checks first: real words, real speaker names, variety."""
    words = re.findall(r"[a-z’]+", sample.lower())
    names = SPEAKER.findall(sample)
    return {"words": len(words),
            "real words": round(sum(w in train_words for w in words) / max(len(words), 1), 2),
            "different words": round(len(set(words)) / max(len(words), 1), 2),
            "speaker lines": len(names),
            "real names": round(sum(n in speakers for n in names) / max(len(names), 1), 2)}


rows = {"valid text (real Shakespeare)": judge(valid_text[:1000]),
        "pair counts (bigram)": judge(bigram_generate("ROMEO.\n", length=1000))}
rows.update({f"transformer, step {s}": judge(samples[s]) for s in SAMPLE_AT[1:]})
rows.update({f"temperature {t}": judge(generate(model, "ROMEO.\n", length=1000, temperature=t, seed=0))
             for t in (0, 0.5, 1.5)})
for name, row in rows.items():
    print(f"{name:30s}", row)

Now ask it a question about a fact in the plays (Macbeth killed King Duncan). Run the cell.

In [ ]:
question = "HORATIO.\nWho killed King Duncan, and why?\n\nHAMLET.\n"
print(generate(model, question, length=150, temperature=0.5, seed=0))

> **Check.** You should see a reply in the shape of a speech, and no answer to the question.

## 10. Save, reload in a new process, and write from your own prompt

Save the weights (`state_dict`) and everything needed to use them: the vocabulary, the settings and the text's checksum. Run the cell.

In [ ]:
import hashlib, json

torch.save(model.state_dict(), "tiny_lm.pt")
info = {"chars": chars, "config": {"dim": 128, "heads": 4, "blocks": 3, "context": 64, "dropout": 0.1},
        "text": "shakespeare_tragedies.txt",
        "text_sha256": hashlib.sha256(Path("shakespeare_tragedies.txt").read_bytes()).hexdigest(),
        "split": "first 90% train, last 10% valid", "steps": STEPS, "seed": 0,
        "valid_loss": round(results["tiny transformer"], 4), "torch": torch.__version__}
Path("tiny_lm.json").write_text(json.dumps(info, ensure_ascii=False, indent=1), encoding="utf-8")
for f in ("tiny_lm.pt", "tiny_lm.json"):
    print(f"{f:13s} {Path(f).stat().st_size / 1000:,.1f} kB")

The next cell writes `generate.py`, a small program that loads the two files and writes text. It contains its own copy of the `TinyLM` class, because a `state_dict` holds only numbers. Run the cell.

In [ ]:
GENERATE_PY = '"""Write text with the saved tiny language model, in a new process.\n\n    python generate.py "PROMPT" [--length 200] [--temperature 0.8] [--seed 0]\n"""\nimport argparse\nimport json\nimport sys\n\nimport torch\nfrom torch import nn\n\nclass TinyLM(nn.Module):\n    """Character + position vectors -> blocks with a causal mask -> a score for every character."""\n\n    def __init__(self, vocab_size, dim=128, heads=4, blocks=3, context=64, dropout=0.1):\n        super().__init__()\n        self.context = context\n        self.embed = nn.Embedding(vocab_size, dim)            # one row of numbers per character\n        self.position = nn.Embedding(context, dim)            # one row of numbers per position\n        block = nn.TransformerEncoderLayer(dim, heads, 4 * dim, dropout=dropout,\n                                           batch_first=True, norm_first=True)\n        self.blocks = nn.TransformerEncoder(block, blocks, enable_nested_tensor=False)\n        self.norm = nn.LayerNorm(dim)\n        self.out = nn.Linear(dim, vocab_size)                 # a score for every character\n\n    def forward(self, ids):                                   # ids: (batch, n)\n        n = ids.shape[1]\n        h = self.embed(ids) + self.position(torch.arange(n))  # (batch, n, dim)\n        mask = nn.Transformer.generate_square_subsequent_mask(n)\n        h = self.blocks(h, mask=mask, is_causal=True)         # each position reads only the past\n        return self.out(self.norm(h))                         # (batch, n, vocab_size)\n\n\nparser = argparse.ArgumentParser()\nparser.add_argument("prompt")\nparser.add_argument("--length", type=int, default=200)\nparser.add_argument("--temperature", type=float, default=0.8)\nparser.add_argument("--seed", type=int, default=0)\nargs = parser.parse_args()\n\ntorch.set_num_threads(2)\ninfo = json.load(open("tiny_lm.json", encoding="utf-8"))\nchars = info["chars"]\nstoi = {c: i for i, c in enumerate(chars)}\nunknown = sorted(set(args.prompt) - set(chars))\nif unknown:\n    sys.exit(f"These characters are not in the vocabulary: {unknown}")\n\nmodel = TinyLM(len(chars), **info["config"])\nmodel.load_state_dict(torch.load("tiny_lm.pt"))\nmodel.eval()\n\ng = torch.Generator().manual_seed(args.seed)\nids = [stoi[c] for c in args.prompt]\nwith torch.no_grad():\n    for _ in range(args.length):\n        scores = model(torch.tensor([ids[-model.context:]]))[0, -1]\n        if args.temperature == 0:\n            next_id = int(scores.argmax())\n        else:\n            probs = torch.softmax(scores / args.temperature, dim=-1)\n            next_id = int(torch.multinomial(probs, 1, generator=g))\n        ids.append(next_id)\nprint("".join(chars[i] for i in ids))\n'
Path("generate.py").write_text(GENERATE_PY, encoding="utf-8")
print("wrote generate.py")

Now run it in a fresh Python process, and compare with the notebook's own `generate()`. Run the cell.

In [ ]:
import subprocess, sys

prompt = "MACBETH.\nWhat news"
run = subprocess.run([sys.executable, "generate.py", prompt, "--length", "150", "--seed", "1"],
                     capture_output=True, text=True, check=True)        # a fresh Python process
print(run.stdout)
print("same as in this notebook:", run.stdout.rstrip("\n") == generate(model, prompt, length=150, seed=1))

> **Check.** The last line should say `same as in this notebook: True`.

> **Your turn.** Write your own prompt in `my_prompt`, for example a speaker's name in capitals, a full stop, a new line and the first words of a speech. Use `’` for an apostrophe, or none. Run `generate.py` with it in a new process, and put the text that it prints in `my_output`. Then run the check cell.

In [ ]:
my_prompt = "JULIET.\nO Romeo, "
run = subprocess.run([sys.executable, "generate.py", my_prompt, "--seed", "7"], capture_output=True, text=True)
my_output = run.stdout or run.stderr
print(my_output)

In [ ]:
if isinstance(my_output, str) and isinstance(my_prompt, str) and my_output.startswith(my_prompt):
    print("Check passed: generate.py continued your prompt.")
else:
    print("Not yet: my_output should start with my_prompt. Use only characters of the vocabulary.")

## Make it a portfolio piece

The lesson page lists what to put in a public repository: a README with the problem, the data and its status, the baselines, the curves, the samples at several steps, and the honest comparison with a real language model. Keep `tiny_lm.json` with it, and say that the text is public domain and comes from Project Gutenberg.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Final assignment](https://learning.nextia-ai.com/courses/deep-learning/final-assignment/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m09/train-a-tiny-language-model/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The text is public domain (Shakespeare, from Project Gutenberg eBook #100, with the Project Gutenberg header and licence removed).